# Chapter 4 — Key Learnings

This notebook contains my main takeaways, definitions, and conceptual notes
from **Chapter 4 - Implementing a GPT Model from scratch to generate text** of *'Build a Large Language Model (From Scratch)'* book by Sebastian Raschka.

### 0. Chapter Objective
Build the main components of a GPT-like decoder-only transformer and connect them into a complete model capable of autoregressive text generation.

### 1. Adding our repo root 'build-llm-from-scratch-pytorch' to sys.path

In [1]:
from pathlib import Path
import sys

# Current folder:
# repository_root/chapter_04/exercises
# i.e. 
# import os  
# print(os.getcwd()) # prints: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch\chapter_04\exercises 
# Note: Python searches for chapter_03 (and all other needed imports) inside that folder and in the other locations listed in sys.path. but sys.path currently does not have the repo root
# the snippet below adds the repo root to sys.path

repo_root = Path.cwd().parents[1]

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

print("Repository root:", repo_root)
sys.path

Repository root: c:\Users\delmi\Documents\LEARNING\Manning_Learning\repos\build-llm-from-scratch-pytorch


['c:\\Users\\delmi\\Documents\\LEARNING\\Manning_Learning\\repos\\build-llm-from-scratch-pytorch',
 'C:\\Users\\delmi\\anaconda3\\python312.zip',
 'C:\\Users\\delmi\\anaconda3\\DLLs',
 'C:\\Users\\delmi\\anaconda3\\Lib',
 'C:\\Users\\delmi\\anaconda3',
 'c:\\Users\\delmi\\venvs\\llmbookvenv',
 '',
 'c:\\Users\\delmi\\venvs\\llmbookvenv\\Lib\\site-packages']

### 2. LayerNorm class

In [2]:
import torch 
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))
    
    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False) # Here we use the 'biased' Population Variance (i.e. /n instead of /n-1)
        norm_x =  (x-mean)/torch.sqrt(var+self.eps) # We add a small positive eps to avoid dividing by zero 
        return self.scale * norm_x + self.shift # Where scale and shift are two trainable parameters that the LLM automatically adjusts during training  

### 3. FeedForward class

In [3]:
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4*cfg["emb_dim"]), # Explands to a higher dim (4 x emb_dim)
            nn.GELU(), # applies a non-liear transformation (GELU) - Not that we could also use our own custom GELU() class 
            nn.Linear(4*cfg["emb_dim"], cfg["emb_dim"]) # contracts back to initial dim: emb_dim
        )

    def forward(self, x):
        return self.layers(x)

### 4. TransformerBlock class

In [4]:
import torch.nn as nn
from chapter_03.scripts.attention import MultiHeadAttention

class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"]
        )
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.ff = FeedForward(cfg)

    def forward(self, x):
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)
        x = self.drop_shortcut(x)
        x = x + shortcut # in the sum, x corresponds to the output, and shortcut corresponds to the initial value of x

        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        x = self.drop_shortcut(x)
        x = x + shortcut
        return  x

### 5. GPTModel class
**Purpose:** Converts **token IDs** into contextual representations and finally into **vocabulary-sized logits** for next-token prediction.

In [5]:
import torch.nn as nn

class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"]) # tok_embedding_layer
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"]) # pos_embedding_layer
        self.drop_emb = nn.Dropout(cfg["drop_rate"]) # Dropout layer with dropout rate p
        self.trf_blocks = nn.Sequential(*[TransformerBlock(cfg) for _ in range(cfg["n_layers"])]) # *: unpacks the Transformer blocks and stacks them a number n_layers of times
        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(cfg["emb_dim"], cfg["vocab_size"], bias=False) # This is the LM Head 

    def forward(self, in_idx): # in_idx: Batch of Input Token IDs with shape (b, num_tokens=seq_len)
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(
            torch.arange(seq_len, device=in_idx.device)
            )
        x = tok_embeds + pos_embeds # this is the Input Embeddings with shape (b, seq_len, emb_dim)
        x = self.drop_emb(x) # shape: (b, seq_len, emb_dim)
        x = self.trf_blocks(x) # shape: (b, seq_len, emb_dim)
        x = self.final_norm(x) # shape: (b, seq_len, emb_dim)
        logits = self.out_head(x) # raw, unnormalized logits - shape: (b, seq_len, vocab_size)
        return logits

### 6. generate_simple_text function

In [6]:
def generate_simple_text(model, idx, max_new_tokens, context_size): # idx shape: (b, curr_seq_len)
    for _ in range(max_new_tokens): # repeats a number 'maxnew_tokens' of iterations
        idx_cond =  idx[:, -context_size:] # idx conditioned context - shape: (b, seq_len)
        with torch.no_grad():
            logits = model(idx_cond) # shape: (b, seq_len, vocab_size)
        logits = logits[:, -1, :] # shape: (b, 1, vocab_size) ~ (b, vocab_size)
        probas = torch.softmax(logits, dim=-1)
        idx_next = torch.argmax(probas, dim=-1, keepdim=True) # This uses 'greedy decoding' (i.e. choose the highest-probability next token)
        idx = torch.cat((idx, idx_next), dim=1) # appends next-token to the sequence
    return idx

**Generation loop:**

1. Keep only the latest `context_size` tokens.
2. Run the model.
3. Select the logits for the last position.
4. Choose the highest-probability next token.
5. Append it to the sequence.
6. Repeat.

### 7. GPT-2 models configs

In [7]:
GPT2_cfg_small = {
    "vocab_size": 50257,
    "context_length": 1024,
    "emb_dim": 768,
    "n_heads": 12,
    "n_layers": 12,
    "drop_rate": 0.1,
    "qkv_bias": False    
}

GPT2_cfg_medium = GPT2_cfg_small.copy()
GPT2_cfg_medium.update({"emb_dim": 1024, "n_heads": 16, "n_layers":24})

GPT2_cfg_large = GPT2_cfg_small.copy()
GPT2_cfg_large.update({"emb_dim": 1280, "n_heads": 20, "n_layers":36})

GPT2_cfg_XL = GPT2_cfg_small.copy()
GPT2_cfg_XL.update({"emb_dim": 1600, "n_heads": 25, "n_layers":48})

### 8. Example

In [8]:
import tiktoken
start_context = "Hello, I am"
tokenizer = tiktoken.get_encoding("gpt2")
encoded = tokenizer.encode(start_context)
print("encoded:", encoded) # [15496, 11, 314, 716]
encoded_tensor = torch.tensor(encoded).unsqueeze(0) # .unsqueeze(0) adds a new dim of size 1 at position 0
print("encoded_tensor.shape:", encoded_tensor.shape) # torch.Size([1, 4])

model = GPTModel(GPT2_cfg_small)
model.eval() # .eval() mode --> disables random components like Dropout which are only used during training

# We use the 'generate_simple_text' function
out = generate_simple_text(model=model, 
                           idx=encoded_tensor,
                           max_new_tokens=6,
                           context_size=GPT2_cfg_small["context_length"])
print("Output:", out)
print("Output length:", len(out[0]))
decode_text = tokenizer.decode(out.squeeze(0).tolist())
print("decode_text:", decode_text)

encoded: [15496, 11, 314, 716]
encoded_tensor.shape: torch.Size([1, 4])
Output: tensor([[15496,    11,   314,   716, 19496,  3023, 11081, 40176, 18686, 41766]])
Output length: 10
decode_text: Hello, I ambach04opy lamented juven Constantin


Note that the decode_text is giberish because our model *has NOT been trained yet*.

### 9. GPT Shape flow


```text
Token IDs
[b, n]

↓ token embedding

[b, n, emb_dim]

+ positional embeddings
[n, emb_dim]

↓

Input embeddings
[b, n, emb_dim]

↓

Transformer blocks
[b, n, emb_dim]

↓

Final LayerNorm
[b, n, emb_dim]

↓

Output head

Logits
[b, n, vocab_size]

↓

Select last-token logits
[b, vocab_size]

↓

Choose next token
[b, 1]
```

### 10. Key definitions
* **Logits:** Raw output scores produced by the model before softmax. There is one logit per vocabulary token.

* **Shortcut connection:** Adds a block's input directly to its transformed output, helping information and gradients flow through deep networks.

* **Layer normalization:** Normalizes each token representation across its embedding dimension.

* **Feed-forward network:** A small neural network applied independently to each token after attention.

* **Autoregressive generation:** Generating one token at a time and feeding previously generated tokens back into the model.

### 11. Q/As

- **Q: What is the purpose of layer normalization in a GPT model?**  
  Layer normalization normalizes each token representation across its embedding dimension, helping stabilize and improve neural network training. 

- **Q: Why are shortcut connections used in transformer blocks?**  
  Shortcut connections add a block’s input directly to its transformed output, which helps maintain gradient flow through deep networks and reduces the vanishing-gradient problem. 

- **Q: What are the main components of a transformer block in GPT?**  
  A transformer block combines masked multi-head attention, layer normalization, dropout, a feed-forward network, GELU activations, and shortcut connections. {index=2}

- **Q: What is the difference between the role of self-attention and the feed-forward network inside a transformer block?**  
  Self-attention identifies and analyzes relationships between tokens in the sequence, whereas the feed-forward network transforms each token representation independently at each position. 

- **Q: Does a transformer block change the embedding dimension of its input?**  
  No. The operations inside the transformer block are designed to preserve the dimensionality of each token representation, so the output has the same overall shape as the input. 

- **Q: How are token embeddings and positional embeddings used in the GPT model?**  
  Tokenized text is converted into token embeddings, which are then augmented with positional embeddings. The resulting combined embeddings are passed through the stack of transformer blocks.  

- **Q: What happens after the final transformer block in GPT?**  
  The output passes through a final layer normalization and then a linear output layer that maps each token representation to a vector whose dimension equals the vocabulary size.  

- **Q: What does the output head of the GPT model produce?**  
  It produces one score, or logit, for every token in the vocabulary at every sequence position, allowing the model to predict the next token.  

- **Q: How many transformer blocks are used in the 124-million-parameter GPT-2 model described in the chapter?**  
  The 124-million-parameter GPT-2 model uses 12 stacked transformer blocks, controlled by the `n_layers` configuration parameter.  

- **Q: Why does an untrained GPT model generate gibberish even though its architecture is complete?**  
  Chapter 4 constructs the GPT architecture but does not yet pretrain it; pretraining is introduced in chapter 5. Therefore, the model’s randomly initialized parameters have not yet learned meaningful language patterns. 